## Paper numbers reproduced by this script
Paper: "The Limits of Query Monitoring for Distributed Model Extraction".
Run on Kaggle (see "Kaggle:" line below for hardware/time). Seeds are fixed in the configuration cell;
re-runs vary by a few points (the paper reports run-to-run spread up to ~0.07-0.14 for weekly detection).

| Paper location | Quantity | Value in paper | Where computed here |
|---|---|---|---|
| App. K (first paragraph) | design: pooled populations, weekly blocks, benign failover team | 250, 2,500 and 10,000 account-weeks; 12 weekly blocks; team of 20 accounts, half of it flagged | Cell 1, `S34` (`AS`, `BLOCKS`, `TEAM`); Cell R, team `G` |
| App. K | admissible throttle schedules | 4,094 | Cell R, `ASSIGN` |
| §8 'The provider's own randomness'; App. J (Prop. 5 paragraph); App. K; Prediction log (Step 34, P34a) | randomisation null on real, drifting traffic: rejection at 1% on null runs | "one rejection at 1% in 120 null runs"; Partial (1/40 at 250; 0/40 at 2,500 and 10,000) | Cell R, scenario `null`, print "[R] A=... null: rejection at 1%"; Cell Z "P34a" print |
| §8 'The provider's own randomness'; App. J; App. K; Prediction log (Step 34, P34b/P34c) | failover campaign (reassigned within the block / one block later) detected | "caught in every run" / "detected in every run"; 1.00 at 250 and 2,500 | Cell R, scenarios `fail0`, `fail1`; Cell Z "P34b/c" print |
| App. K; Prediction log (Step 34, P34b) | throttles at A = 10,000: flagged group already holds all 50 colluders | Fail (0.00: all 50 colluders already flagged) | Cell R, scenario `fail0` at A=10000, `reject` and `colluders_in_G`; Cell Z "P34b/c" print |
| App. J; App. K | throughput lost by a campaign that dropped throttled work | 40% | Cell R, scenario `nofail`, `throughput_lost` |
| §8 'The provider's own randomness'; App. J; App. K; Prediction log (Step 34, P34d) | benign shared-queue team with failover fires the test | 82–98% of runs; Pass (0.82–0.98) | Cell R, scenario `team`; Cell Z "P34d" print |
| App. K | legitimate accounts throttled per detection, at deployed precision 0.73 and 0.54 measured at 5,000–7,552 account-weeks | about 16–42 | not computed here (paper combines this design with the precision measured in Step 32) |
| App. K; Prediction log (Step 34, P34e) | toy answer marks: adaptive campaign at 10% uptake within four weeks; fixed-prompt campaign; answer-sharing benign team | 0.62; never (0.00); as often (identical) | Cell M, `RESM` / print "[M] ..."; Cell Z "P34e" print |
| Prediction log (Step 34, P34f); App. F 'Statistics that failed' | dual-reference test (slope of frozen − lagged score) detecting ramps | 0.03 | Cell D, `det_slope`, print "[D] ramp detection"; Cell Z "P34f" print |
| Prediction log (Step 34, P34f); App. D.6 'Teaching the monitor that the campaign is normal' | reference frozen from early weeks, alone: ramp detection and false alarms | 0.44 at 5.2% | Cell D, `det_frz` with `FA_F` / `FA_F_DRIFT` in print "[D] ramp detection" (which of the two false-alarm figures is the 5.2%: not isolated) |

The comparison value 0.12 (ramped campaign against the lagged reference alone, App. D.6; quoted in P34f below) comes from Step 21, not from this script.

Not reported in the paper: QUICK-mode outputs; `nofail` rejection rates; per-A colluder counts in the flagged group and `all_colluders_flagged`; answer-mark detection at uptake 0.02/0.05/0.20 and at 1 or 2 weeks; pooled leave-one-out false alarms of the slope test (`FA_S`); the number of null sequences; the JSON rows of the verdict report.

# Step 34 —Randomized throttles (exact randomization null), answer marks, and a dual reference against ramped campaigns
**Kaggle:** GPU + Internet + `HF_TOKEN` → **Run All** (≈60–90 min; reuses earlier caches). `QUICK=True` first. Everything here is a **simulation** on WildChat traffic: public logs contain no interventions, so failover and adaptivity are modelled and stated.

* **R — randomized throttles (Proposition 5).** Pooled populations of A = 250 / 2,500 / 10,000 account-weeks (weeks ≤ 43, so each has 12 later weeks of its own recorded traffic). A 50-account MetaMathQA campaign (1,640 queries/week) is injected in block 0; the flagged group G is the multi-outlier group; the response set B is the 50 accounts most correlated with G in the centred hollowed matrix (fixed before randomizing). G is throttled in a random half of 12 weekly blocks. Failover models: work reassigned to the other colluders within the block (`fail0`), one block later (`fail1`), or dropped (`nofail`). Statistic: max over lags 0 and 1 of the difference in B's on-task volume between throttled and unthrottled blocks; exact p-value over all 2¹² assignments. Nulls: no campaign (benign B's recorded future traffic, with real drift and churn). Confound: a benign team of 20 accounts working on the task (benign on-task queries, e.g. an evaluation pipeline) that shares a queue and fails over, with 10 of them flagged. The response is on-task volume, so a benign team working off task would not respond.
* **M — answer marks (toy).** Each answer carries one of k = 4 equivalent random follow-up suggestions. Adaptive colluders draw later queries from the latest suggestion shown to another colluder with probability u; a fixed-set campaign ignores suggestions; a benign team shares answers the same way; ordinary users only follow their own. Statistic: within the flagged group, the number of queries that follow a suggestion shown to another member; null by redrawing all marks from the design.
* **D — dual reference.** The ramped campaign (volume rising linearly over the 8 weeks before a test week) is scored against the contaminated lagged reference and a reference frozen from weeks 0–7; the slope over 8 weeks of the difference (frozen − lagged two-scale score) is tested against slopes of null sequences. A frozen-only detector is reported with its false alarms.

| ID | Locked prediction (locked in this notebook; outcomes in the prediction log (docs/prediction_log.md), Step 34) |
|---|---|
| **P34a** | Randomization p-values are valid on real benign traffic: rejection ≤ 1.5% at α = 1% on null runs, at every A |
| **P34b** | Throttles detect a failover campaign at A = 10,000 (0.5% colluders): power ≥ 0.5 with reassignment within the block |
| **P34c** | Delayed failover is still visible: power ≥ 0.2 with reassignment one block late |
| **P34d** | Benign shared-queue teams fire the test: rejection ≥ 0.5 when the flagged group is a failover team (an expected limit) |
| **P34e** | Answer marks: adaptive campaign (u = 10%) detected within 4 weeks ≥ 0.8; fixed-set ≤ 0.02; answer-sharing benign team detected as often |
| **P34f** | Dual reference: ramp detection ≥ 0.5 (0.12 with the lagged reference alone), false alarms ≤ 10% on null weeks |

In [ ]:
# ===== Cell 1: configuration =====
import math, json, time, os, gc, re, pickle
import numpy as np
QUICK = False
SYNTHETIC_TEST = False
SEED = 9091
OUTDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
try:
    import torch
    HAS_TORCH = True; DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except Exception:
    HAS_TORCH, DEV = False, None
USE_GPU = HAS_TORCH and DEV.type == "cuda"
if not USE_GPU and not SYNTHETIC_TEST:
    raise RuntimeError("No GPU visible. Kaggle: Settings -> Accelerator -> GPU, then restart.")
CHUNK = 32768 if USE_GPU else 8192
H_COARSE, H_FINE, M_MODES, ALPHA, FLOOR = 4.0, 0.5, 300, 0.01, 1e-4
VOL_CAP, MIN_PER_ACCT, POOL_FRAC, SUB_FRAC = 500, 10, 0.2, 0.7
N_TOPICS, N_REF_BATCH, REF_BATCH, MMD_WIN, MINCELL = 100, 20, 500, 2000, 20
DETS = ["twoscale", "qc", "qcount", "mmd", "gmmd", "glob", "spec", "fraudar"]
SRC_MAIN = ["metamath_unique", "metamath_mixed", "metamath_followup", "gsm8k_human", "random"]
EVALS = [("eval_gsm8k", 3, 300), ("eval_gsm8k", 10, 100), ("eval_mmlu", 3, 300), ("eval_mmlu", 10, 100)]
SRC_EXTRA = ["wild_cluster"]
CAMO_FRAC = 0.2
if QUICK:
    WK = dict(KS=[20], CAL_N=4, LAGB=2, N_TEST=3, NULL_PER=12, REPS=2)
    POP = dict(AS=[40, 70], K=6, N_FIX=200, STEP=4, FIRST=12, N_CAL=12, N_TEST=6, REPS=3)
    EVALS = [("eval_gsm8k", 3, 40), ("eval_mmlu", 3, 40)]
    MMD_WIN, N_REF_BATCH, REF_BATCH = 400, 5, 200
else:
    WK = dict(KS=[50], CAL_N=8, LAGB=4, N_TEST=24, NULL_PER=40, REPS=4)
    POP = dict(AS=[250, 500, 1000], K=50, N_FIX=1640, STEP=4, FIRST=20, N_CAL=40, N_TEST=20, REPS=10)
CACHE11 = f"{OUTDIR}/step11_wildchat_E384_{'quick' if QUICK else 'full'}{'_synth' if SYNTHETIC_TEST else ''}.npz"
CACHE13 = f"{OUTDIR}/step13_distill_{'quick' if QUICK else 'full'}{'_synth' if SYNTHETIC_TEST else ''}.npz"
CACHE14 = f"{OUTDIR}/step14_meta_{'quick' if QUICK else 'full'}{'_synth' if SYNTHETIC_TEST else ''}.npz"
CACHE15 = f"{OUTDIR}/step15_extra_{'quick' if QUICK else 'full'}{'_synth' if SYNTHETIC_TEST else ''}.npz"
N_CONV = 30000 if QUICK else None; DIMS = [384]
print(f"backend {'GPU ' + torch.cuda.get_device_name(0) if USE_GPU else 'CPU'} | QUICK={QUICK}")
CACHE18 = f"{OUTDIR}/step18_evals_{'quick' if QUICK else 'full'}{'_synth' if SYNTHETIC_TEST else ''}.npz"
import contextlib
@contextlib.contextmanager
def modes(m):
    global M_MODES
    old = M_MODES; M_MODES = m
    try: yield
    finally: M_MODES = old
if QUICK: S27 = dict(K=20, REPS=2, AS=[30, 60, 120], NUL={30: 25, 60: 20, 120: 15}, ATT=6, KP=6, NP=200, A_HI=60, M_HI=150, NUL_HI=15, ATT_HI=6, PF_N=300)
else:     S27 = dict(K=50, REPS=4, AS=[250, 1000, 2500, 5000, 10000], NUL={250: 150, 1000: 120, 2500: 100, 5000: 80, 10000: 50}, ATT=30, KP=50, NP=1640, A_HI=2500, M_HI=750, NUL_HI=100, ATT_HI=30, PF_N=4000)
# [PAPER] App. K: pooled populations of 250, 2,500 and 10,000 account-weeks; 12 weekly blocks; benign team of 20 accounts
if QUICK: S34 = dict(AS=[40, 80], RUNS=6, BLOCKS=8, KP=6, NP=200, TEAM=8, RAMP_TW=2, RAMP_REPS=3, NULL_REPS=2, M_REPS=40)
else:     S34 = dict(AS=[250, 2500, 10000], RUNS=40, BLOCKS=12, KP=50, NP=1640, TEAM=20, RAMP_TW=6, RAMP_REPS=10, NULL_REPS=5, M_REPS=400)

## Part 0 — caches

In [ ]:
# ===== Cell 2: full 384-d MiniLM embeddings of English user turns (cached as float16) =====
if os.path.exists(CACHE11):
    _c = np.load(CACHE11); E16, ACCT, WEEK = _c["E16"], _c["ACCT"], _c["WEEK"]; print(f"loaded cache: {E16.shape}")
else:
    t0 = time.time()
    if SYNTHETIC_TEST:
        rng = np.random.default_rng(0); n_ip, n_rows = 3000, 60000
        ACCT = rng.integers(0, n_ip, n_rows).astype(np.int32); WEEK = rng.integers(0, 30, n_rows).astype(np.int16)
        topic = rng.integers(0, 40, n_ip)[ACCT]; base = rng.standard_normal((40, 384))
        E = (base[topic] * 0.8 + rng.standard_normal((n_rows, 384)) * 0.6).astype(np.float32)
    else:
        from datasets import load_dataset
        tok = None
        try:
            from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = os.environ.get("HF_TOKEN")
        split = "train" if N_CONV is None else f"train[:{N_CONV}]"
        ds = load_dataset("allenai/WildChat-1M", split=split, token=tok).select_columns(["conversation", "hashed_ip", "timestamp", "language"])
        texts, ips, tss = [], [], []
        for batch in ds.iter(batch_size=2000):
            for conv, ip, ts, lang in zip(batch["conversation"], batch["hashed_ip"], batch["timestamp"], batch["language"]):
                if lang != "English" or ip is None or ts is None: continue
                for turn in conv:
                    if turn.get("role") == "user" and turn.get("content"):
                        texts.append(turn["content"][:1000]); ips.append(ip); tss.append(ts)
        print(f"{len(texts)} English user turns from {len(set(ips))} IPs ({time.time()-t0:.0f}s)")
        _, ACCT = np.unique(np.array(ips), return_inverse=True); ACCT = ACCT.astype(np.int32)
        days = np.array([np.datetime64(t.replace(tzinfo=None) if hasattr(t, "replace") else t, "D") for t in tss])
        WEEK = ((days - days.min()).astype(np.int64) // 7).astype(np.int16); del ds; gc.collect()
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            os.system("pip install -q sentence-transformers"); from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda"); enc.max_seq_length = 256
        E = enc.encode(texts, batch_size=512, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
        del texts; gc.collect()
    E16 = E.astype(np.float16); del E; gc.collect()
    np.savez(CACHE11, E16=E16, ACCT=ACCT, WEEK=WEEK); print(f"cached ({time.time()-t0:.0f}s)")
n_w = int(WEEK.max()) + 1
# PCA basis on a 100k fit sample (nested: the first d components of the same basis for every d)
rng = np.random.default_rng(SEED); fit = E16[rng.choice(len(E16), min(100000, len(E16)), replace=False)].astype(np.float32)
MU_E = fit.mean(0); ew, U = np.linalg.eigh(np.cov(fit - MU_E, rowvar=False)); EW, UB = ew[::-1], U[:, ::-1].copy()
for d in DIMS: print(f"d={d:3d}: explains {EW[:d].sum()/EW.sum():.3f} of embedding variance")
def make_Z(d):
    W_ = UB[:, :d] / math.sqrt(EW[:d].sum() / 4.0); Z = np.empty((len(E16), d), dtype=np.float32)
    for s0 in range(0, len(E16), 200000): Z[s0:s0 + 200000] = (E16[s0:s0 + 200000].astype(np.float32) - MU_E) @ W_
    return Z

In [ ]:
# ===== Cell 3: conversation ids, turn indices and attacker-pool text pairs (same order as the cache), MetaMathQA pairs =====
import pickle
rng = np.random.default_rng(SEED + 1); n_acct = int(ACCT.max()) + 1; is_pool = rng.random(n_acct) < POOL_FRAC    # identical to Steps 12-13
PAIRS_PKL = CACHE14.replace(".npz", "_pairs.pkl")
if os.path.exists(CACHE14) and os.path.exists(PAIRS_PKL):
    _c = np.load(CACHE14); CONV, TURN, D_MMU = _c["CONV"], _c["TURN"], _c["D_MMU"]
    with open(PAIRS_PKL, "rb") as fh: POOL_TXT, MM_PAIRS = pickle.load(fh)
    print(f"loaded metadata cache: {len(CONV)} turns, {len(POOL_TXT)} pool pairs, {len(MM_PAIRS)} MetaMathQA pairs")
else:
    t0 = time.time()
    if SYNTHETIC_TEST:
        r = np.random.default_rng(5); CONV = np.cumsum(r.random(len(E16)) < 0.4).astype(np.int32); TURN = np.ones(len(E16), np.int16)
        for i in range(1, len(E16)):
            if CONV[i] == CONV[i - 1]: TURN[i] = TURN[i - 1] + 1
        POOL_TXT = {int(i): (f"user prompt {i}", f"assistant reply {i} " * 5) for i in np.flatnonzero(is_pool[ACCT])}
        MM_PAIRS = [(f"What is {a} plus {a+1}?", f"{a} + {a+1} = {2*a+1}. The answer is: {2*a+1}") for a in range(1500)]
        D_MMU = (np.random.default_rng(3).standard_normal(384) * 2 + np.random.default_rng(4).standard_normal((1500, 384)) * 0.7).astype(np.float16)
    else:
        from datasets import load_dataset
        tok = None
        try:
            from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = os.environ.get("HF_TOKEN")
        split = "train" if N_CONV is None else f"train[:{N_CONV}]"
        ds = load_dataset("allenai/WildChat-1M", split=split, token=tok).select_columns(["conversation", "hashed_ip", "timestamp", "language"])
        ips, conv_id, turn_ix, users, replies = [], [], [], [], []; cid = 0
        for batch in ds.iter(batch_size=2000):
            for conv, ip, ts, lang in zip(batch["conversation"], batch["hashed_ip"], batch["timestamp"], batch["language"]):
                if lang != "English" or ip is None or ts is None: continue
                t = 0
                for j, turn in enumerate(conv):
                    if turn.get("role") == "user" and turn.get("content"):
                        t += 1; nxt = next((c.get("content") or "" for c in conv[j + 1:] if c.get("role") == "assistant"), "")
                        ips.append(ip); conv_id.append(cid); turn_ix.append(t); users.append(turn["content"][:1000]); replies.append(nxt[:1500])
                cid += 1
        _, ACCT2 = np.unique(np.array(ips), return_inverse=True)
        assert len(ACCT2) == len(ACCT) and np.array_equal(ACCT2.astype(np.int32), ACCT), "re-read order does not match the Step 11 cache"
        CONV, TURN = np.array(conv_id, np.int32), np.array(turn_ix, np.int16)
        POOL_TXT = {int(i): (users[i], replies[i]) for i in np.flatnonzero(is_pool[ACCT])}
        del users, replies, ds; gc.collect()
        print(f"re-read WildChat: order verified, {len(CONV)} turns in {int(CONV.max())+1} conversations, {len(POOL_TXT)} pool pairs ({time.time()-t0:.0f}s)")
        mm = load_dataset("meta-math/MetaMathQA", split="train", token=tok).select_columns(["query", "response", "original_question"])
        groups = {}
        for q, a, o in zip(mm["query"], mm["response"], mm["original_question"]):
            if q and a and o: groups.setdefault(o, []).append((q.strip()[:1000], a.strip()[:2000]))
        rr = np.random.default_rng(SEED); origs = list(groups); rr.shuffle(origs)
        MM_PAIRS = [groups[o][int(rr.integers(len(groups[o])))] for o in origs]          # one (query, response) per source problem
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            os.system("pip install -q sentence-transformers"); from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda"); enc.max_seq_length = 256
        D_MMU = enc.encode([q for q, _ in MM_PAIRS], batch_size=512, convert_to_numpy=True, normalize_embeddings=True).astype(np.float16)
        print(f"MetaMathQA: {len(MM_PAIRS)} (query, response) pairs, one per source problem ({time.time()-t0:.0f}s)")
    np.savez(CACHE14, CONV=CONV, TURN=TURN, D_MMU=D_MMU)
    with open(PAIRS_PKL, "wb") as fh: pickle.dump((POOL_TXT, MM_PAIRS), fh)
DIST = {"metamath_unique": D_MMU}
print(f"follow-up turns (turn >= 2) among all turns: {np.mean(TURN >= 2):.3f}")

In [ ]:
# ===== Cell 3: MetaMathQA grouped by source problem (unique vs rephrased) + Magicoder, MiniLM, cached =====
if os.path.exists(CACHE13):
    _c = np.load(CACHE13); DIST13 = {k[2:]: _c[k] for k in _c.files if k.startswith("D_")}; MM_GROUP = _c["MM_GROUP"]; print({k: v.shape for k, v in DIST13.items()})
else:
    t0 = time.time(); DIST13 = {}
    if SYNTHETIC_TEST:
        r = np.random.default_rng(3); c = r.standard_normal(384) * 2
        base = c + r.standard_normal((1500, 384)) * 0.7
        DIST13["metamath_unique"] = base.astype(np.float16)
        reph = np.repeat(base[:600], 4, axis=0) + r.standard_normal((2400, 384)) * 0.05; DIST13["metamath_rephrase"] = reph.astype(np.float16)
        MM_GROUP = np.repeat(np.arange(600), 4)
        DIST13["magicoder"] = (r.standard_normal(384) * 2 + r.standard_normal((2000, 384)) * 0.7).astype(np.float16)
    else:
        from datasets import load_dataset
        tok = None
        try:
            from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = os.environ.get("HF_TOKEN")
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            os.system("pip install -q sentence-transformers"); from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda"); enc.max_seq_length = 256
        emb = lambda T: enc.encode(T, batch_size=512, convert_to_numpy=True, normalize_embeddings=True).astype(np.float16)
        mm = load_dataset("meta-math/MetaMathQA", split="train", token=tok).select_columns(["query", "original_question"])
        qs, oq = mm["query"], mm["original_question"]; groups = {}
        for q, o in zip(qs, oq):
            if q and o: groups.setdefault(o, set()).add(q.strip()[:1000])
        rng = np.random.default_rng(SEED); origs = list(groups); rng.shuffle(origs)
        uniq = [sorted(groups[o])[int(rng.integers(len(groups[o])))] for o in origs]
        multi = [o for o in origs if len(groups[o]) >= 4][:6000]
        reph, gid = [], []
        for j, o in enumerate(multi):
            for q in rng.permutation(sorted(groups[o]))[:4]: reph.append(q); gid.append(j)
        DIST13["metamath_unique"] = emb(uniq); DIST13["metamath_rephrase"] = emb(reph); MM_GROUP = np.array(gid)
        print(f"MetaMathQA: {len(uniq)} source problems (unique set), {len(multi)} problems x 4 variants (rephrase set) ({time.time()-t0:.0f}s)")
        mg = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train", token=tok)
        idx = np.random.default_rng(SEED).choice(len(mg), min(25000, len(mg)), replace=False)
        DIST13["magicoder"] = emb(list(dict.fromkeys(str(mg[int(i)]["problem"])[:1000] for i in idx)))
        print(f"Magicoder: {len(DIST13['magicoder'])} prompts ({time.time()-t0:.0f}s)")
    np.savez(CACHE13, MM_GROUP=MM_GROUP, **{f"D_{k}": v for k, v in DIST13.items()})

In [ ]:
# ===== Cell 4: extra sources — human GSM8K questions, templated MetaMathQA, natural-likeness scores (cached) =====
if os.path.exists(CACHE15):
    _c = np.load(CACHE15); D_GSM, D_TMPL, NAT = _c["D_GSM"], _c["D_TMPL"], _c["NAT"]
else:
    t0 = time.time()
    if SYNTHETIC_TEST:
        r = np.random.default_rng(11); D_GSM = (D_MMU[:1000].astype(np.float32) + r.standard_normal((1000, 384)) * 0.3).astype(np.float16)
        D_TMPL = (D_MMU.astype(np.float32) * 0.7 + 0.3).astype(np.float16)
    else:
        from datasets import load_dataset
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            os.system("pip install -q sentence-transformers"); from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda"); enc.max_seq_length = 256
        emb = lambda T: enc.encode(T, batch_size=512, convert_to_numpy=True, normalize_embeddings=True).astype(np.float16)
        g = load_dataset("openai/gsm8k", "main", split="train")
        D_GSM = emb([q[:1000] for q in g["question"]])                                    # human-written math word problems
        TEMPLATE = "You are a careful math tutor. Solve the following problem step by step and state the final answer.\n\n"
        D_TMPL = emb([TEMPLATE + q for q, _ in MM_PAIRS])                                   # same problems, fixed template prefix
    # natural-likeness of each MetaMathQA prompt: mean cosine similarity to its 10 nearest benign WildChat turns
    ben = np.flatnonzero(~is_pool[ACCT]); smp = np.random.default_rng(3).choice(ben, min(50000, len(ben)), replace=False)
    B_ = E16[smp].astype(np.float32); Q_ = D_MMU.astype(np.float32); NAT = np.empty(len(Q_), np.float32)
    for s0 in range(0, len(Q_), 2048):
        if USE_GPU:
            sims = torch.as_tensor(Q_[s0:s0 + 2048], device=DEV) @ torch.as_tensor(B_, device=DEV).T
            NAT[s0:s0 + 2048] = sims.topk(10, dim=1).values.mean(1).cpu().numpy()
        else:
            sims = Q_[s0:s0 + 2048] @ B_.T; NAT[s0:s0 + 2048] = np.sort(sims, 1)[:, -10:].mean(1)
    np.savez(CACHE15, D_GSM=D_GSM, D_TMPL=D_TMPL, NAT=NAT); print(f"extra sources built ({time.time()-t0:.0f}s)")
CAMO = np.flatnonzero(NAT >= np.quantile(NAT, 1 - CAMO_FRAC))
DIST = {"metamath_unique": D_MMU, "metamath_rephrase": DIST13["metamath_rephrase"], "gsm8k_human": D_GSM, "metamath_template": D_TMPL}
print(f"sources: " + ", ".join(f"{k}:{len(v)}" for k, v in DIST.items()) + f" | camouflage pool {len(CAMO)} (natural-likeness >= {np.quantile(NAT, 1 - CAMO_FRAC):.3f}; all MetaMathQA median {np.median(NAT):.3f})")
# ---- benchmark sweeps for the benign evaluation-pipeline test (GSM8K test questions, MMLU questions)
if os.path.exists(CACHE18):
    _c = np.load(CACHE18); D_EG, D_EM = _c["D_EG"], _c["D_EM"]
else:
    if SYNTHETIC_TEST:
        r = np.random.default_rng(21); D_EG = (D_GSM[:300].astype(np.float32) + r.standard_normal((300, 384)) * 0.2).astype(np.float16)
        D_EM = (r.standard_normal(384) * 2 + r.standard_normal((800, 384)) * 0.6).astype(np.float16)
    else:
        from datasets import load_dataset
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            os.system("pip install -q sentence-transformers"); from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda"); enc.max_seq_length = 256
        emb = lambda T: enc.encode(T, batch_size=512, convert_to_numpy=True, normalize_embeddings=True).astype(np.float16)
        D_EG = emb([q[:1000] for q in load_dataset("openai/gsm8k", "main", split="test")["question"]])
        mm = load_dataset("cais/mmlu", "all", split="test")
        D_EM = emb([(q + "\n" + "\n".join(f"{chr(65+i)}. {c}" for i, c in enumerate(ch)))[:1000] for q, ch in zip(mm["question"], mm["choices"])])
    np.savez(CACHE18, D_EG=D_EG, D_EM=D_EM)
DIST["eval_gsm8k"], DIST["eval_mmlu"] = D_EG, D_EM
print(f"evaluation sweeps: GSM8K test {len(D_EG)}, MMLU {len(D_EM)}")

## Statistics

In [ ]:
# ===== Cell 4: projection, topics, natural source cluster with farthest-point order =====
from sklearn.cluster import MiniBatchKMeans
rng = np.random.default_rng(SEED + 1); n_acct = int(ACCT.max()) + 1; is_pool = rng.random(n_acct) < POOL_FRAC; pool_idx = np.flatnonzero(is_pool[ACCT])
Z = make_Z(384); W384 = UB[:, :384] / math.sqrt(EW[:384].sum() / 4.0); EN = E16
DZ = {k: ((v.astype(np.float32) - MU_E) @ W384).astype(np.float32) for k, v in DIST.items()}
t0 = time.time(); ben = np.flatnonzero(~is_pool[ACCT])
km = MiniBatchKMeans(n_clusters=N_TOPICS, random_state=SEED, batch_size=4096, n_init=3).fit(Z[np.random.default_rng(1).choice(ben, min(200000, len(ben)), replace=False)])
TOPIC = km.predict(Z).astype(np.int32)
CL10 = MiniBatchKMeans(n_clusters=10, random_state=SEED + 1, batch_size=4096, n_init=3).fit(Z[pool_idx]).labels_
WILD = pool_idx[CL10 == int(np.argmax(np.bincount(CL10)))]
def fps(X, N, seed=0):
    N = min(N, len(X)); r = np.random.default_rng(seed); i0 = int(r.integers(len(X)))
    if USE_GPU:
        Xt = torch.as_tensor(X, dtype=torch.float32, device=DEV); d2 = ((Xt - Xt[i0]) ** 2).sum(1); sel = torch.empty(N, dtype=torch.long, device=DEV); sel[0] = i0
        for t in range(1, N): i = torch.argmax(d2); sel[t] = i; d2 = torch.minimum(d2, ((Xt - Xt[i]) ** 2).sum(1))
        return sel.cpu().numpy()
    sel = [i0]; d2 = ((X - X[i0]) ** 2).sum(1)
    for _ in range(N - 1): i = int(np.argmax(d2)); sel.append(i); d2 = np.minimum(d2, ((X - X[i]) ** 2).sum(1))
    return np.array(sel)
WILD_FPS = WILD[fps(Z[WILD], min(len(WILD), 20000), seed=SEED)]
_sub = EN[np.random.default_rng(2).choice(len(EN), 2000, replace=False)].astype(np.float32)
_d2 = ((_sub[:1000, None, :] - _sub[None, 1000:, :]) ** 2).sum(-1); MMD_BW = float(np.sqrt(np.median(_d2) / 2)); del _d2
print(f"natural source cluster {len(WILD)} prompts (farthest-point order {len(WILD_FPS)}); sources: " + ", ".join(f"{k}:{len(v)}" for k, v in DIST.items()) + f" ({time.time()-t0:.0f}s)")

In [ ]:
# ===== Cell 5: augmented real whitening, statistics, MMD, Fraudar =====
def make_bank_aug(Zref, h, seed):
    r = np.random.default_rng(seed); modes = r.standard_normal((M_MODES, Zref.shape[1])) / h / (2 * np.pi); m2 = 2 * M_MODES
    if USE_GPU:
        Mt = torch.as_tensor(modes, dtype=torch.float64, device=DEV); G = torch.zeros((m2, m2), dtype=torch.float64, device=DEV); mu = torch.zeros(m2, dtype=torch.float64, device=DEV)
        for s0 in range(0, len(Zref), 16384):
            th = (2 * math.pi) * (torch.as_tensor(Zref[s0:s0 + 16384], dtype=torch.float64, device=DEV) @ Mt.T)
            X = torch.cat([torch.cos(th), torch.sin(th)], 1); G += X.T @ X; mu += X.sum(0)
        G, mu = G.cpu().numpy(), mu.cpu().numpy()
    else:
        G = np.zeros((m2, m2)); mu = np.zeros(m2)
        for s0 in range(0, len(Zref), 4096):
            th = 2 * np.pi * (Zref[s0:s0 + 4096] @ modes.T); X = np.hstack([np.cos(th), np.sin(th)]); G += X.T @ X; mu += X.sum(0)
    G /= len(Zref); mu /= len(Zref); Sig = G - np.outer(mu, mu); ew, U = np.linalg.eigh((Sig + Sig.T) / 2); keep = ew > FLOOR * ew.max()
    B = dict(modes=modes, mu=mu, W=U[:, keep] / np.sqrt(ew[keep]), r=int(keep.sum()))
    if USE_GPU:
        B["MT"] = torch.as_tensor(modes, dtype=torch.float64, device=DEV); B["MU_T"] = torch.as_tensor(mu, dtype=torch.float64, device=DEV); B["W_T"] = torch.as_tensor(B["W"], dtype=torch.float64, device=DEV)
    return B
def whitened(pts, lab, cnt, B):
    if USE_GPU:
        Fc = torch.zeros((len(cnt), M_MODES), dtype=torch.float64, device=DEV); Fs = torch.zeros_like(Fc)
        for s0 in range(0, len(pts), CHUNK):
            P = torch.as_tensor(pts[s0:s0 + CHUNK], dtype=torch.float64, device=DEV); ix = torch.as_tensor(lab[s0:s0 + CHUNK], dtype=torch.long, device=DEV)
            th = (2.0 * math.pi) * (P @ B["MT"].T); Fc.index_add_(0, ix, torch.cos(th)); Fs.index_add_(0, ix, torch.sin(th))
        c = torch.as_tensor(cnt, dtype=torch.float64, device=DEV)
        return (((torch.cat([Fc, Fs], 1) - c[:, None] * B["MU_T"][None, :]) / torch.sqrt(c)[:, None]) @ B["W_T"]).cpu().numpy()
    import scipy.sparse as sp
    X = np.zeros((len(cnt), 2 * M_MODES))
    for s0 in range(0, len(pts), CHUNK):
        P, l_ = pts[s0:s0 + CHUNK], lab[s0:s0 + CHUNK]; th = 2 * np.pi * (P @ B["modes"].T)
        X += sp.csr_matrix((np.ones(len(P)), (l_, np.arange(len(P)))), shape=(len(cnt), len(P))) @ np.hstack([np.cos(th), np.sin(th)])
    return ((X - cnt[:, None] * B["mu"][None, :]) / np.sqrt(cnt)[:, None]) @ B["W"]
def union_S(Y, cnt, ix): return float(((Y[ix] * np.sqrt(cnt[ix])[:, None]).sum(0) ** 2).mean() / cnt[ix].sum())
def mmd2(Xw, refs):
    # Liu-Guo-Dong style: unbiased MMD^2 between window sample and each benign reference batch, averaged
    def k(A_, B_):
        if USE_GPU: return torch.exp(-torch.cdist(A_, B_) ** 2 / (2 * MMD_BW ** 2))
        d2 = (A_ ** 2).sum(1)[:, None] + (B_ ** 2).sum(1)[None, :] - 2 * A_ @ B_.T; return np.exp(-np.maximum(d2, 0) / (2 * MMD_BW ** 2))
    X = torch.as_tensor(Xw, dtype=torch.float32, device=DEV) if USE_GPU else Xw
    n = len(X); Kxx = k(X, X); sxx = (Kxx.sum() - Kxx.diagonal().sum()) / (n * (n - 1)); out = []
    for R in refs:
        m = len(R); Kyy = k(R, R); Kxy = k(X, R)
        out.append(float(sxx + (Kyy.sum() - Kyy.diagonal().sum()) / (m * (m - 1)) - 2 * Kxy.mean()))
    return float(np.mean(out))
def fraudar(lab, top, A_):
    C = np.zeros((A_, N_TOPICS)); np.add.at(C, (lab, top), 1.0); wt = 1.0 / np.log(C.sum(0) + 5.0); Wm = C * wt[None, :]
    da, dt = Wm.sum(1), Wm.sum(0); alive_a, alive_t = np.ones(A_, bool), np.ones(N_TOPICS, bool); tot = Wm.sum(); n = A_ + N_TOPICS; best = tot / n
    for _ in range(n - 1):
        ia = np.where(alive_a, da, np.inf).argmin(); it = np.where(alive_t, dt, np.inf).argmin()
        if da[ia] <= dt[it]:
            alive_a[ia] = False; tot -= da[ia]; dt -= Wm[ia] * alive_t; da[ia] = np.inf
        else:
            alive_t[it] = False; tot -= dt[it]; da -= Wm[:, it] * alive_a; dt[it] = np.inf
        n -= 1; best = max(best, tot / max(n, 1))
    return float(best)
def window_stats(idx_pts, lab, cnt, Bc, Bf, refs, extra=None, en_extra=None, top_extra=None):
    pts = Z[idx_pts] if extra is None else np.vstack([extra[0], Z[idx_pts]]); labs = lab if extra is None else np.concatenate([extra[1], lab])
    en = EN[idx_pts] if extra is None else np.vstack([en_extra, EN[idx_pts]]); top = TOPIC[idx_pts] if extra is None else np.concatenate([top_extra, TOPIC[idx_pts]])
    Yc = whitened(pts, labs, cnt, Bc); Yf = whitened(pts, labs, cnt, Bf); A_ = len(cnt)
    ev, V = np.linalg.eigh(Yc.T @ Yc / Bc["r"]); v = Yc @ V[:, -1]; v2 = v ** 2 / (v ** 2).sum()
    o = np.argsort(-v2); m = int(np.clip(np.searchsorted(np.cumsum(v2[o]), 0.5) + 1, 3, 100)); grp = o[:m]
    r2 = (Yc ** 2).mean(1); rr = np.random.default_rng(len(labs)); samp = rr.choice(len(en), min(MMD_WIN, len(en)), replace=False)
    return dict(spec=float(ev[-1]), r2=r2, acct=float(r2.max()), glob=union_S(Yc, cnt, np.arange(A_)),
                mmd=mmd2(en[samp].astype(np.float32), refs), fraudar=fraudar(labs, top, A_),
                vol=float(np.sort(cnt)[-10:].sum() / cnt.sum()), twoscale=float(np.log(union_S(Yc, cnt, grp)) - np.log(union_S(Yf, cnt, grp))), v2=v2, grp=grp)
def hc_stat(x, ns):
    M_ = len(ns); p = np.sort((1 + M_ - np.searchsorted(ns, x, side="left")) / (M_ + 1)); A_ = len(p); k = max(1, A_ // 2)
    i = np.arange(1, k + 1); pp = p[:k]; return float(np.max(np.sqrt(A_) * (i / A_ - pp) / np.sqrt(pp * (1 - pp))))

In [ ]:
# ===== Cell 7: sources, query-cluster statistics, extended window statistics =====
FOLLOW = np.array([i for i in POOL_TXT if TURN[i] >= 2]) if len(POOL_TXT) else pool_idx
def _mm(key, ix): return DZ[key][ix], DIST[key][ix].astype(np.float32), km.predict(DZ[key][ix]).astype(np.int32)
def campaign_points(src, n, r):
    if src == "random": ix = r.choice(pool_idx, n, replace=False); return Z[ix], EN[ix].astype(np.float32), TOPIC[ix]
    if src == "wild_cluster": ix = r.choice(WILD, min(n, len(WILD)), replace=n > len(WILD)); return Z[ix], EN[ix].astype(np.float32), TOPIC[ix]
    if src in ("metamath_unique", "gsm8k_human", "metamath_template", "eval_gsm8k", "eval_mmlu"):
        D = DIST[src]; return _mm(src, r.choice(len(D), min(n, len(D)), replace=n > len(D)))
    if src == "metamath_camouflage": return _mm("metamath_unique", r.choice(CAMO, min(n, len(CAMO)), replace=n > len(CAMO)))
    if src == "metamath_rephrase":
        g = r.permutation(int(MM_GROUP.max()) + 1)[: int(math.ceil(n / 4))]; ix = np.flatnonzero(np.isin(MM_GROUP, g))[:n]; return _mm(src, ix)
    if src == "metamath_repeat":
        half = (n + 1) // 2; ix = r.choice(len(D_MMU), half, replace=False); ix = np.repeat(ix, 2)[:n]; return _mm("metamath_unique", ix)
    nm = (n + 1) // 2; ix = r.choice(len(D_MMU), nm, replace=False); other = FOLLOW if src == "metamath_followup" else pool_idx   # mixed / follow-up
    jx = r.choice(other, n - nm, replace=False); a, b, c = _mm("metamath_unique", ix)
    Zs, Es, Ts = np.empty((n, Z.shape[1]), np.float32), np.empty((n, EN.shape[1]), np.float32), np.empty(n, np.int32)
    Zs[0::2], Es[0::2], Ts[0::2] = a, b, c; Zs[1::2], Es[1::2], Ts[1::2] = Z[jx], EN[jx].astype(np.float32), TOPIC[jx]
    return Zs, Es, Ts                                                     # prompt and its padding/follow-up land in the same account

def cell_stats(pts, top, Bc, Bf):
    # two-scale score per topic cell, pooling queries across all accounts
    u, inv = np.unique(top, return_inverse=True); cnt = np.bincount(inv)
    yc = (whitened(pts, inv, cnt, Bc) ** 2).mean(1); yf = (whitened(pts, inv, cnt, Bf) ** 2).mean(1)
    ts = np.full(N_TOPICS, np.nan); n = np.zeros(N_TOPICS); n[u] = cnt; ok = cnt >= MINCELL
    ts[u[ok]] = np.log(yc[ok]) - np.log(yf[ok]); return ts, n / n.sum()
def window_stats15(idx_pts, lab, cnt, Bc, Bf, refs, extra=None, en_extra=None, top_extra=None):
    s = window_stats(idx_pts, lab, cnt, Bc, Bf, refs, extra=extra, en_extra=en_extra, top_extra=top_extra)
    pts = Z[idx_pts] if extra is None else np.vstack([extra[0], Z[idx_pts]]); labs = lab if extra is None else np.concatenate([extra[1], lab])
    en = EN[idx_pts] if extra is None else np.vstack([en_extra, EN[idx_pts]]); top = TOPIC[idx_pts] if extra is None else np.concatenate([top_extra, TOPIC[idx_pts]])
    s["cell_ts"], s["cell_share"] = cell_stats(pts, top, Bc, Bf)
    gm = np.isin(labs, s["grp"]); ge = en[gm]
    if len(ge) > MMD_WIN: ge = ge[np.random.default_rng(len(labs)).choice(len(ge), MMD_WIN, replace=False)]
    s["gmmd"] = mmd2(ge.astype(np.float32), refs) if len(ge) >= 20 else 0.0
    s["labs"] = labs; s["top"] = top; s["n_total"] = len(top); return s
def hc_stat(x, ns):
    M_ = len(ns); p = np.sort((1 + M_ - np.searchsorted(ns, x, side="left")) / (M_ + 1)); A_ = len(p); k = max(1, A_ // 2)
    i = np.arange(1, k + 1); pp = p[:k]; return float(np.max(np.sqrt(A_) * (i / A_ - pp) / np.sqrt(pp * (1 - pp))))
def calibrate15(recs):
    half = recs[0::2]; rest = recs[1::2] if len(recs) > 1 else recs
    TS = np.array([r["cell_ts"] for r in half]); SH = np.array([r["cell_share"] for r in half]); NT = np.array([r["n_total"] for r in half], float)
    LN = np.log(np.maximum(SH * NT[:, None], 1.0)); valid = np.mean(~np.isnan(TS), 0) >= 0.5
    A_ = np.zeros(N_TOPICS); B_ = np.zeros(N_TOPICS); SC = np.ones(N_TOPICS)
    for t in np.flatnonzero(valid):
        ok = ~np.isnan(TS[:, t]); x, y = LN[ok, t], TS[ok, t]
        b_, a_ = np.polyfit(x, y, 1) if np.ptp(x) > 0 else (0.0, float(np.median(y)))
        res = y - (a_ + b_ * x); A_[t], B_[t] = a_, b_; SC[t] = max(1.4826 * np.median(np.abs(res - np.median(res))), 0.05)
    med_sh = np.median(SH, 0); sc_sh = np.maximum(1.4826 * np.median(np.abs(SH - med_sh), 0), 0.002 + 0.1 * med_sh)
    T = dict(A=A_, B=B_, SC=SC, valid=valid, med_sh=med_sh, sc_sh=sc_sh)
    for r in recs: add_scores(r, T)
    q_ = lambda a: float(np.quantile(a, 1 - ALPHA))
    for d in DETS: T[d] = q_([r[d] for r in rest])
    return T
def add_scores(s, T):
    ln = np.log(np.maximum(s["cell_share"] * s["n_total"], 1.0))
    with np.errstate(all="ignore"):
        z = np.where(T["valid"] & ~np.isnan(s["cell_ts"]), (s["cell_ts"] - (T["A"] + T["B"] * ln)) / T["SC"], np.nan)
    s["qc"] = float(np.nanmax(z)) if np.any(~np.isnan(z)) else -1e9; s["qc_cell"] = int(np.nanargmax(z)) if np.any(~np.isnan(z)) else -1
    s["qcount"] = float(np.max((s["cell_share"] - T["med_sh"]) / T["sc_sh"])); return s
def fires15(s, T): return {d: s[d] > T[d] for d in DETS}
def boot_ci(x, B=1000, rng=np.random.default_rng(7)):
    x = np.array(x, float); bs = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(B)]; return float(x.mean()), float(np.quantile(bs, .025)), float(np.quantile(bs, .975))
def roc(null_vals, att_vals):
    null_vals, att_vals = np.array(null_vals), np.array(att_vals)
    auc = float(np.mean(att_vals[:, None] > null_vals[None, :]) + 0.5 * np.mean(att_vals[:, None] == null_vals[None, :]))
    return dict(auc=auc, tpr1=float(np.mean(att_vals > np.quantile(null_vals, 0.99))), tpr5=float(np.mean(att_vals > np.quantile(null_vals, 0.95))))

In [ ]:
# ===== Cell 8: spectral machinery for R_c (Gram form), multi-outlier selection, diagnostics =====
from functools import lru_cache
@lru_cache(maxsize=64)
def mp_median(c):
    a, b = (1 - np.sqrt(c)) ** 2, (1 + np.sqrt(c)) ** 2; x = np.linspace(a + 1e-9, b - 1e-9, 20001)
    f = np.sqrt(np.maximum((b - x) * (x - a), 0)) / (2 * np.pi * c * x); F = np.cumsum(f) * (x[1] - x[0]); F /= F[-1]
    return float(x[np.searchsorted(F, 0.5)])
def top_group(v):
    v2 = v ** 2 / (v ** 2).sum(); o = np.argsort(-v2); m = int(np.clip(np.searchsorted(np.cumsum(v2[o]), 0.5) + 1, 3, 100)); return o[:m]
def ts_of(Yc, Yf, cnt, g): return float(np.log(union_S(Yc, cnt, g)) - np.log(union_S(Yf, cnt, g)))
def gram(Yn, kk=50):
    A_, r_ = Yn.shape
    if A_ >= r_:
        ev, U = np.linalg.eigh(Yn.T @ Yn); o = np.argsort(-ev); ev, U = ev[o], U[:, o]
        V = Yn @ U[:, :kk] / np.sqrt(np.maximum(ev[:kk], 1e-12)); return ev, V
    ev, V = np.linalg.eigh(Yn @ Yn.T); o = np.argsort(-ev); return ev[o], V[:, o][:, :kk]
def stats27(pts, labs, cnt, Bc, Bf, top=None, K_or=None, cells=False):
    Yc = whitened(pts, labs, cnt, Bc); Yf = whitened(pts, labs, cnt, Bf); A_, r_ = Yc.shape
    ev, Vc = np.linalg.eigh(Yc.T @ Yc / r_); gC = top_group(Yc @ Vc[:, -1])
    Ycc = Yc - Yc.mean(0, keepdims=True); Yn = Ycc / np.maximum(np.linalg.norm(Ycc, axis=1, keepdims=True), 1e-12)
    lam, V = gram(Yn); nz = lam[: min(A_, r_)]; g_ = A_ / r_
    e_hat = float(np.median(nz) / (max(g_, 1.0) * mp_median(min(g_, 1 / g_))) * (1 + np.sqrt(g_)) ** 2 - 1)
    J, kpre = [0], 0
    for j in range(min(20, V.shape[1])):
        if lam[j] - 1 > 1.05 * e_hat:
            kpre += 1
            if j > 0 and 1.0 / np.sum(V[:, j] ** 4) >= 5: J.append(j)
    gR = top_group(V[:, 0]); groups = [top_group(V[:, j]) for j in J]; sc = [ts_of(Yc, Yf, cnt, g) for g in groups]; jm = int(np.argmax(sc))
    s = dict(tsC=ts_of(Yc, Yf, cnt, gC), tsRc=sc[0], Tmax=float(max(sc)), k=len(J) - 1, kpre=kpre, lam1=float(lam[0] - 1), ehat=e_hat, gR=gR, gT=groups[jm])
    rr = np.random.default_rng(len(labs))
    def mcos(g): G_ = Yn[g] @ Yn[g].T; n_ = len(g); return float((G_.sum() - n_) / max(n_ * (n_ - 1), 1))
    s["cos_sel"] = mcos(gR); s["cos_rand"] = float(np.mean([mcos(rr.choice(A_, len(gR), replace=False)) for _ in range(5)]))
    if K_or:
        e = np.zeros(A_); e[:K_or] = 1 / np.sqrt(K_or); ov = (V.T @ e) ** 2; js = int(np.argmax(ov))
        s.update(jstar=js, ov=float(ov[js]), lam_js=float(lam[js] - 1), rho=mcos(np.arange(K_or)), orc=ts_of(Yc, Yf, cnt, np.arange(K_or)))
    if cells and top is not None:
        s["cell_ts"], s["cell_share"] = cell_stats(pts, top, Bc, Bf); s["n_total"] = len(top)
    return s
DETW = ["tsC", "tsRc", "Tmax"]

## R — randomized throttles

In [ ]:
# ===== Cell R: randomized throttles with an exact randomization null =====
import itertools
from sklearn.linear_model import LogisticRegression
t0 = time.time()
neg = np.flatnonzero((~is_pool[ACCT]) & (WEEK < 8)); neg = np.random.default_rng(4).choice(neg, min(4 * len(D_MMU), len(neg)), replace=False)
clf_t = LogisticRegression(max_iter=2000, class_weight="balanced").fit(np.vstack([D_MMU.astype(np.float32), EN[neg].astype(np.float32)]), np.r_[np.ones(len(D_MMU)), np.zeros(len(neg))])
ONT = np.zeros(len(EN), bool)
for s0 in range(0, len(EN), 200000): ONT[s0:s0 + 200000] = clf_t.predict_proba(EN[s0:s0 + 200000].astype(np.float32))[:, 1] >= 0.5
n_b = int(WEEK.max()) + 1; T_ = S34["BLOCKS"]
order5 = np.lexsort((ACCT, WEEK)); keys5 = WEEK[order5].astype(np.int64) * n_acct + ACCT[order5]
AW = {}
for g in np.split(order5, np.flatnonzero(np.diff(keys5)) + 1):
    if not is_pool[int(ACCT[g[0]])]: AW[(int(ACCT[g[0]]), int(WEEK[g[0]]))] = g[:VOL_CAP]
POOLR = [(a, w) for (a, w), g in AW.items() if len(g) >= 5 and w + T_ < n_b]
refR = np.random.default_rng(12).choice(np.concatenate([AW[k] for k in POOLR[:: max(1, len(POOLR) // 20000)]]), 100000 if not QUICK else 5000, replace=True)
BR = (make_bank_aug(Z[refR], H_COARSE, SEED + 7), make_bank_aug(Z[refR], H_FINE, SEED + 8))
def future_ontask(key, t):
    g = AW.get((key[0], key[1] + t)); return int(ONT[g].sum()) if g is not None else 0
def expansion(pts, labs, cnt, G, k=50):
    Yc = whitened(pts, labs, cnt, BR[0]); Ycc = Yc - Yc.mean(0, keepdims=True); Yn = Ycc / np.maximum(np.linalg.norm(Ycc, axis=1, keepdims=True), 1e-12)
    corr = Yn @ Yn[G].mean(0); corr[G] = -np.inf; return np.argsort(-corr)[:k]
# [PAPER] App. K: 4,094 admissible schedules (at least one throttled and one unthrottled block)
ASSIGN = np.array([z for z in itertools.product([0, 1], repeat=T_) if 0 < sum(z) < T_], dtype=float)
def rand_p(Z0, V):
    def stat(Zm):
        out = []
        for lag in (0, 1):
            Zl = np.roll(Zm, lag, axis=-1); Zl[..., :lag] = 0
            n1 = np.maximum(Zl.sum(-1), 1); n0 = np.maximum((1 - Zl).sum(-1), 1); out.append((Zl * V).sum(-1) / n1 - ((1 - Zl) * V).sum(-1) / n0)
        return np.maximum(*out)
    return float(np.mean(stat(ASSIGN) >= stat(Z0[None, :])[0] - 1e-12))
RESR = []
for A_ in S34["AS"]:
    if A_ > 0.8 * len(POOLR): print(f"[R] A={A_} exceeds pool; skipped"); continue
    for scen in ["null", "fail0", "fail1", "nofail", "team"]:
        rej, hit_col, loss = [], [], []
        for run in range(S34["RUNS"]):
            r = np.random.default_rng(SEED + 1000 * A_ + 37 * run + len(scen)); pick = r.choice(len(POOLR), A_, replace=False); keys = [POOLR[i] for i in pick]
            parts = [AW[k] for k in keys]; vols = np.array([len(p) for p in parts]); idx = np.concatenate(parts); lab = np.repeat(np.arange(A_), vols)
            K_ = S34["KP"] if scen in ("fail0", "fail1", "nofail") else (S34["TEAM"] if scen == "team" else 0)
            if K_ and scen != "team":
                cv = r.choice(vols, K_, replace=True).astype(float); cv = np.maximum(1, np.round(cv * S34["NP"] / cv.sum())).astype(int)
                Zc, _, _ = campaign_points("metamath_unique", int(cv.sum()), r); n = len(Zc); labc = np.repeat(np.arange(K_), cv)[:n]
            elif scen == "team":
                pool_cell = np.flatnonzero(ONT & (~is_pool[ACCT]))                      # a benign team working on the task (e.g. an evaluation pipeline with failover)
                if len(pool_cell) < 50: pool_cell = np.flatnonzero(~is_pool[ACCT])[: 5000]
                cv = r.choice(vols, K_, replace=True); jx = r.choice(pool_cell, int(cv.sum()), replace=len(pool_cell) < int(cv.sum())); Zc = Z[jx]; n = len(Zc); labc = np.repeat(np.arange(K_), cv)[:n]
            if K_:
                keep = lab >= K_; _, inv = np.unique(np.concatenate([labc, lab[keep]]), return_inverse=True); pts = np.vstack([Zc, Z[idx[keep]]]); cnt = np.bincount(inv)
                bkeys = [None] * K_ + [keys[i] for i in range(A_) if i >= K_]
            else:
                pts, inv, cnt = Z[idx], lab, vols; bkeys = keys
            s = stats27(pts, inv, cnt, *BR)
            G = np.array(sorted(r.choice(K_, K_ // 2, replace=False))) if scen == "team" else np.asarray(s["gT"])
            B = expansion(pts, inv, cnt, G); col = np.arange(K_) if K_ else np.array([], int)
            hit_col.append(float(np.isin(G, col).sum()))
            Zs = (r.random(T_) < 0.5).astype(float)
            if Zs.sum() in (0, T_): Zs[0] = 1 - Zs[0]
            V = np.zeros(T_)
            for t in range(T_):
                V[t] += sum(future_ontask(bkeys[b], t + 1) for b in B if bkeys[b] is not None)
            if K_:
                base = cnt[:K_].astype(float) if scen != "team" else cnt[:K_].astype(float)
                gc_ = np.intersect1d(G, col); others = np.setdiff1d(col, G); inB = np.intersect1d(B, others)
                LG = base[gc_].sum(); share = (len(inB) / max(len(others), 1)) if len(others) else 0.0
                for t in range(T_):
                    V[t] += base[inB].sum() * (1.0 if scen != "team" else ONT[jx].mean())
                    src_t = t if scen in ("fail0", "team") else t - 1
                    if scen in ("fail0", "fail1", "team") and src_t >= 0 and Zs[src_t] == 1: V[t] += LG * share * (1.0 if scen != "team" else ONT[jx].mean())
                # [PAPER] App. J, App. K: a campaign that dropped throttled work lost 40% of its throughput
                if scen == "nofail": loss.append(float(LG * Zs.mean() / max(base.sum(), 1)))
            rej.append(float(rand_p(Zs, V) <= ALPHA))
        # [PAPER] §8, App. J, App. K, Prediction log (Step 34): "null" -> one rejection at 1% in 120 null runs (1/40 at 250; 0/40 at 2,500 and 10,000)
        # [PAPER] §8, App. J, App. K, Prediction log (Step 34): "fail0"/"fail1" -> detected in every run (1.00 at 250 and 2,500); 0.00 at 10,000 (all 50 colluders already flagged)
        # [PAPER] §8, App. J, App. K, Prediction log (Step 34): "team" -> benign failover team fires the test in 82–98% of runs
        row = dict(A=A_, scenario=scen, reject=float(np.mean(rej)), colluders_in_G=float(np.mean(hit_col)), all_colluders_flagged=float(np.mean([h >= K_ for h in hit_col])) if K_ else float('nan'), throughput_lost=float(np.mean(loss)) if loss else float("nan"))
        RESR.append(row); print(f"[R] A={A_:6d} {scen:>7}: rejection at 1% = {row['reject']:.3f} | colluders (or team members) in flagged group {row['colluders_in_G']:.1f}" + (f" | throughput lost {row['throughput_lost']:.2f}" if loss else "") + f"  ({time.time()-t0:.0f}s)")

## M — answer marks (toy)

In [ ]:
# ===== Cell M: answer marks (toy simulation) =====
def marks_sim(kind, u, weeks, rng, A_m=200, g=20, q=30, k=4, u_b=0.3):
    steps = weeks * q; marks = rng.integers(1, k + 1, size=(A_m, steps)); f = np.zeros((A_m, steps), int)
    for s_ in range(1, steps):
        own = marks[:, s_ - 1]; f[:, s_] = np.where(rng.random(A_m) < u_b, own, 0)
        if kind in ("adaptive", "team"):
            for b in range(g):
                if rng.random() < u:
                    a = rng.integers(g - 1); a = a + (a >= b); f[b, s_] = marks[a, s_ - 1]
    def stat(M):
        tot = 0
        for b in range(g):
            fb = f[b, 1:]; others = np.delete(M[:g, :-1], b, axis=0); tot += int(np.sum((fb[None, :] == others) & (fb[None, :] > 0)))
        return tot
    obs = stat(marks); null = [stat(rng.integers(1, k + 1, size=marks.shape)) for _ in range(99)]
    return (1 + sum(x >= obs for x in null)) / 100.0
RESM = []
for kind in ["adaptive", "fixed", "team"]:
    for u in ([0.02, 0.05, 0.1, 0.2] if kind != "fixed" else [0.0]):
        for weeks in [1, 2, 4]:
            ps = [marks_sim(kind, u, weeks, np.random.default_rng(SEED + 7 * i + int(100 * u) + weeks)) for i in range(max(10, S34["M_REPS"] // 10))]
            # [PAPER] App. K, Prediction log (Step 34): 4 weeks -> adaptive u=0.10 detected 0.62; fixed never (0.00); benign team identical
            RESM.append(dict(kind=kind, u=u, weeks=weeks, power=float(np.mean(np.array(ps) <= 0.01))))
            print(f"[M] {kind:>8} uptake {u:.2f} weeks {weeks}: detection at 1% = {RESM[-1]['power']:.2f}")

## D — dual reference

In [ ]:
# ===== Cell D: dual reference against the ramped campaign =====
t0 = time.time(); blk = WEEK.astype(np.int32)
order = np.lexsort((ACCT, blk)); keys = blk[order].astype(np.int64) * n_acct + ACCT[order]; GA = {b: [] for b in range(n_b)}
for g in np.split(order, np.flatnonzero(np.diff(keys)) + 1):
    if not is_pool[int(ACCT[g[0]])] and len(g) >= MIN_PER_ACCT: GA[int(blk[g[0]])].append(g[:VOL_CAP])
def sub_window(b, r):
    k = max(20, int(SUB_FRAC * len(GA[b]))); pick = r.choice(len(GA[b]), k, replace=False); parts = [GA[b][i] for i in pick]
    vols = np.array([len(p) for p in parts]); return np.concatenate(parts), np.repeat(np.arange(len(parts)), vols), vols
frz = np.concatenate([np.concatenate(GA[v]) for v in range(0, 8) if GA[v]]); frz = np.random.default_rng(3).choice(frz, min(100000, len(frz)), replace=False)
BF = (make_bank_aug(Z[frz], H_COARSE, SEED + 7), make_bank_aug(Z[frz], H_FINE, SEED + 8))
def lag_bank(v, camp=None):
    prev = [np.concatenate(GA[w]) for w in range(max(0, v - WK["LAGB"]), v) if GA[w]]
    ref = np.random.default_rng(v).choice(np.concatenate(prev), 100000 if not QUICK else 5000, replace=True); X = Z[ref]
    if camp is not None and len(camp): X = np.vstack([X, camp])
    return (make_bank_aug(X, H_COARSE, SEED + 7), make_bank_aug(X, H_FINE, SEED + 8))
def ts2(pts, lab, cnt, B_):
    Yc = whitened(pts, lab, cnt, B_[0]); Yf = whitened(pts, lab, cnt, B_[1]); ev, V = np.linalg.eigh(Yc.T @ Yc / Yc.shape[1]); g = top_group(Yc @ V[:, -1])
    return ts_of(Yc, Yf, cnt, g)
ok_w = [b for b in range(16, n_b) if all(len(GA[v]) >= 40 for v in range(b - 7, b + 1))]
TW = [ok_w[i] for i in np.linspace(0, len(ok_w) - 1, min(S34["RAMP_TW"], len(ok_w))).astype(int)]
# null differences per (week, rep) and null slopes over 8-week sequences
DN = {}
for v in sorted(set(w for b in ok_w for w in range(b - 7, b + 1))):
    Bl = lag_bank(v)
    for rep in range(S34["NULL_REPS"]):
        idx, lab, vols = sub_window(v, np.random.default_rng(SEED + 100 * v + rep)); DN[(v, rep)] = (ts2(Z[idx], lab, vols, BF), ts2(Z[idx], lab, vols, Bl))
def slope(x): return float(np.polyfit(np.arange(len(x)), x, 1)[0])
NS = [slope([DN[(v, rep)][0] - DN[(v, rep)][1] for v in range(b - 7, b + 1)]) for b in ok_w for rep in range(S34["NULL_REPS"])]
FRZ_NULL = [DN[(b, rep)][0] for b in ok_w for rep in range(S34["NULL_REPS"])]
print(f"[D] null sequences {len(NS)} ({time.time()-t0:.0f}s)")
det_slope, det_frz, fa_frz = [], [], []
thr_s = float(np.quantile(NS, 0.99)); thr_f = float(np.quantile(FRZ_NULL, 0.99))
for b in TW:
    camp_ref = campaign_points("metamath_unique", 2 * S34["NP"], np.random.default_rng(b))[0]
    for rep in range(S34["RAMP_REPS"]):
        d = []
        for v in range(b - 7, b + 1):
            frac = 1 - (b - v) / 8.0; r = np.random.default_rng(SEED + 9_700 + 100 * v + rep); idx, lab, vols = sub_window(v, r)
            ncamp_ref = int(np.mean([max(0.0, 1 - (b - w) / 8.0) for w in range(v - WK["LAGB"], v)]) * S34["NP"])
            Bl = lag_bank(v, camp_ref[: max(0, min(ncamp_ref, len(camp_ref)))])
            K_ = S34["KP"]; vol = max(K_, int(frac * S34["NP"]))
            if K_ >= (lab.max() + 1) // 2: d.append(ts2(Z[idx], lab, vols, BF) - ts2(Z[idx], lab, vols, Bl)); continue
            cv = r.choice(vols, K_, replace=True).astype(float); cv = np.maximum(1, np.round(cv * vol / cv.sum())).astype(int)
            Zc, _, _ = campaign_points("metamath_unique", int(cv.sum()), r); n = len(Zc); labc = np.repeat(np.arange(K_), cv)[:n]; keep = lab >= K_
            _, inv = np.unique(np.concatenate([labc, lab[keep]]), return_inverse=True); pts = np.vstack([Zc, Z[idx[keep]]]); cnt = np.bincount(inv)
            tf, tl = ts2(pts, inv, cnt, BF), ts2(pts, inv, cnt, Bl); d.append(tf - tl)
            if v == b: det_frz.append(float(tf > thr_f))
        det_slope.append(float(slope(d) > thr_s))
    print(f"[D] week {b}: slope detection so far {np.mean(det_slope):.2f}, frozen-only {np.mean(det_frz):.2f}  ({time.time()-t0:.0f}s)")
loo = []
for i, x in enumerate(NS):
    rest = np.delete(np.array(NS), i); loo.append(float(x > np.quantile(rest, 0.99)))
FA_S = float(np.mean(loo)); FA_F = float(np.mean([x > np.quantile(np.delete(np.array(FRZ_NULL), i), 0.99) for i, x in enumerate(FRZ_NULL)]))
# frozen-only false alarms on later weeks, thresholds from the earliest third of weeks (the realistic deployment of a frozen reference)
early = [DN[(b, rep)][0] for b in ok_w[: max(1, len(ok_w) // 3)] for rep in range(S34["NULL_REPS"])]; late = [DN[(b, rep)][0] for b in ok_w[len(ok_w) // 3:] for rep in range(S34["NULL_REPS"])]
FA_F_DRIFT = float(np.mean(np.array(late) > np.quantile(early, 0.99)))
# [PAPER] Prediction log (Step 34), App. F 'Statistics that failed': dual-reference slope detection of ramps 0.03
# [PAPER] Prediction log (Step 34), App. D.6: frozen reference alone 0.44 at 5.2% false alarms (whether FA_F or FA_F_DRIFT: not isolated)
print(f"[D] ramp detection: slope of (frozen - lagged) {np.mean(det_slope):.2f} (null false alarms {FA_S:.3f}); frozen-only {np.mean(det_frz):.2f} (false alarms pooled {FA_F:.3f}, with early-period thresholds on later weeks {FA_F_DRIFT:.3f})")

## Predictions and VERDICT REPORT

In [ ]:
# ===== Cell Z: predictions and verdict =====
# [PAPER] Prediction log (Step 34): P34a Partial (1/40; 0/40), P34b Fail (0.00 at 10,000; 1.00 at 250 and 2,500), P34c Pass (1.00), P34d Pass (0.82–0.98)
for A_ in S34["AS"]:
    rr = {r["scenario"]: r for r in RESR if r["A"] == A_}
    if not rr: continue
    print(f"P34a A={A_}: null rejection {rr['null']['reject']:.3f} (<= 0.015)")
    print(f"P34b/c A={A_}: fail0 {rr['fail0']['reject']:.2f} (>= 0.5 at 10,000), fail1 {rr['fail1']['reject']:.2f} (>= 0.2), nofail {rr['nofail']['reject']:.2f} (throughput lost {rr['nofail']['throughput_lost']:.2f}); colluders in G {rr['fail0']['colluders_in_G']:.1f}")
    print(f"P34d A={A_}: benign failover team flagged -> rejection {rr['team']['reject']:.2f} (>= 0.5)")
ga = lambda kind, u, w: next(r["power"] for r in RESM if r["kind"] == kind and r["u"] == u and r["weeks"] == w)
# [PAPER] Prediction log (Step 34): P34e Fail (0.62; 0.00; identical); P34f Fail (slope 0.03; frozen alone 0.44 at 5.2%)
print(f"P34e answer marks, 4 weeks: adaptive u=0.10 {ga('adaptive', 0.1, 4):.2f} (>= 0.8), fixed {ga('fixed', 0.0, 4):.2f} (<= 0.02), benign team u=0.10 {ga('team', 0.1, 4):.2f}")
print(f"P34f dual reference: ramp detection {np.mean(det_slope):.2f} (>= 0.5), null false alarms {FA_S:.3f} (<= 0.10); frozen-only {np.mean(det_frz):.2f}, drift false alarms {FA_F_DRIFT:.3f}")
print("=" * 90); print("STEP 34 VERDICT REPORT"); print("=" * 90)
for r in RESR: print("[R]", json.dumps(r))
for r in RESM: print("[M]", json.dumps(r))
print(f"[D] slope {np.mean(det_slope):.3f} FA {FA_S:.3f} | frozen {np.mean(det_frz):.3f} FA pooled {FA_F:.3f} FA drift {FA_F_DRIFT:.3f}"); print("=" * 90)